# OOP — Classes, Traits, Case Classes & Enums



## `class` — the basics

A class in Scala 3 declares a type and the way to construct instances of that type. The shape:

```scala
class Customer(name: String, city: String):
  // body — fields, methods
```

The parameters in parentheses after the class name form the **primary constructor**. They are not fields by default — they are just constructor parameters, visible inside the class body. To turn them into fields, prefix with `val` (immutable, public read) or `var` (mutable, public read+write).

Visibility defaults to **public**. Use `private` or `protected` to restrict — those work just like in java.

In [ ]:
// Plain class — constructor params are NOT fields
class CustomerA(name: String, city: String):
  def greet: String = s"hello, $name from $city"

val a = CustomerA("alice", "delhi")
println(a.greet)
// println(a.name)  // would not compile — not a field

// val/var on params turns them into fields
class CustomerB(val name: String, var city: String):
  def greet: String = s"hello, $name from $city"

val b = CustomerB("bob", "mumbai")
println(b.name)       // bob — readable
b.city = "bengaluru"  // mutable
println(b.greet)

// Auxiliary constructors with `def this(...)`
class CustomerC(val name: String, val city: String):
  def this(name: String) = this(name, "unknown")

println(CustomerC("carol", "pune").city)  // pune
println(CustomerC("dave").city)            // unknown

## Visibility and methods

`private` and `protected` work as you'd expect. `private` is the strongest — accessible only within the class. `protected` is accessible within the class and subclasses.

Scala adds one nuance worth knowing: `private[this]` makes a field accessible only from the *exact same instance*, not from other instances of the same class. Almost never needed in everyday code, mentioned so it doesn't surprise you.

In [ ]:
class Account(initialBalance: Double):
  private var balance: Double = initialBalance

  def deposit(amount: Double): Unit =
    require(amount > 0, "deposit must be positive")
    balance += amount

  def withdraw(amount: Double): Boolean =
    if amount <= balance then
      balance -= amount
      true
    else
      false

  def currentBalance: Double = balance

val acc = Account(100.0)
acc.deposit(50.0)
println(acc.withdraw(30.0))   // true
println(acc.currentBalance)   // 120.0
// println(acc.balance)       // would not compile — private

## `object` — singletons and companions

An `object` declares a *single instance* of a class — no constructor, no `new` needed, the object simply *is*. Use it for utility methods, configuration, and constants that don't have any per-instance state.

When an `object` has the **same name** as a class in the same file, it is called the **companion object** of that class. Companion objects are special — they can access the class's `private` members, and vice versa. This is where you put factory methods (`apply`) and shared utilities.

The `apply` method on a companion is the basis of the `List(1, 2, 3)` idiom — `List` (no `new`) is calling `List.apply(1, 2, 3)`. Define your own `apply` and your class gets the same call-site sugar.

In [ ]:
class Money private (val cents: Long):
  override def toString: String = f"$$${cents / 100}%d.${cents % 100}%02d"

object Money:
  // Factory method — call as Money(...) without `new`
  def apply(dollars: Int, cents: Int): Money =
    new Money(dollars * 100L + cents)

  // Companion can see Money's private constructor
  def fromCents(c: Long): Money = new Money(c)

  // Constants and shared utilities live here
  val zero: Money = new Money(0)

val price = Money(12, 50)
println(price)               // $12.50
println(Money.zero)          // $0.00
println(Money.fromCents(99)) // $0.99

## `class` vs `object` vs `def` — when to use each

A common early confusion. Three things that all "hold code," chosen by the question they answer.

- **`def`** — *a piece of behaviour*. Pure or impure; called from somewhere. Stateless. Lives on a class, a trait, an object, or at the top level.
- **`class`** — *a way to make many instances, each with its own state*. Has a constructor; you say `MyClass(...)` to make one.
- **`object`** — *exactly one of this thing*. No constructor. Existed before anyone called it.

If you're tempted to put a `def` in a class but the class has no fields and you're never going to make more than one of it — make it an `object` instead. If you're tempted to make an `object` with mutable state shared across the whole program, ask whether that state really belongs anywhere; global state is the same problem in every language.

## `trait` — composable behaviour

A trait is Scala's primary tool for sharing behaviour between unrelated types. Think of it as java's `interface` plus default methods plus fields plus state — but you can mix in multiple traits into one class, which java cannot do.

The shape:

```scala
trait Named:
  def name: String                       // abstract — subclasses must define
  def greet: String = s"hello, $name"    // concrete — inherited automatically
```

You mix a trait in with `extends` for the first and `with` for the rest:

```scala
class Person(val name: String) extends Named with Aged with ...
```

In Scala 3 you can also write `extends X, Y, Z` with commas. Both forms are equivalent.

In [ ]:
trait Named:
  def name: String
  def greet: String = s"hello, $name"

trait Aged:
  def age: Int
  def isAdult: Boolean = age >= 18

class Person(val name: String, val age: Int) extends Named, Aged

val p = Person("ganesh", 30)
println(p.greet)     // hello, ganesh
println(p.isAdult)   // true

// Traits can have fields too
trait Counted:
  private var n = 0
  def tick(): Unit = n += 1
  def count: Int   = n

class EventStream extends Counted
val s = EventStream()
s.tick(); s.tick(); s.tick()
println(s.count)     // 3

## Trait linearization — the one rule for multiple inheritance

When a class mixes in multiple traits and several of them define the same method, which one wins? The answer is **linearization** — Scala arranges all the supertypes in a single ordered list, from most-specific to least-specific, and method lookup walks that list.

The rule, simplified: **left-to-right declaration order is reversed in the linearization, and `super` follows it**. So when you write `extends A, B, C`, the lookup order is roughly `C, B, A, AnyRef, Any`. A `super.method()` call from a class extending `A, B, C` will call the implementation in `C`.

This sounds fiddly and rarely matters in practice — most of your traits will have non-overlapping methods, and when they do overlap you usually want the most-recently-mixed-in version. Mentioned here so the keyword does not surprise you in a Spark or library source file.

In [ ]:
trait Greeter:
  def greet: String = "hi"

trait Loud extends Greeter:
  override def greet: String = super.greet + "!"

trait Excited extends Greeter:
  override def greet: String = super.greet + "!!"

class A extends Greeter, Loud, Excited
class B extends Greeter, Excited, Loud

println(A().greet)   // hi!!!  — Excited wraps Loud wraps base
println(B().greet)   // hi!!!  — Loud wraps Excited wraps base, same total but built differently

## `abstract class` versus `trait`

Both can have abstract members and concrete members. So when do you reach for one over the other?

- **`trait`** — when you want composition; multiple traits can be mixed into one class.
- **`abstract class`** — when you need a *primary constructor with parameters* (Scala 2 traits couldn't take parameters; Scala 3 traits can, with some restrictions), or when you're interoperating with java code that expects a single inheritance hierarchy.

The default is `trait`. Reach for `abstract class` only when you specifically need its features.

## `case class` — the headline feature

A `case class` is a class with a pile of free behaviour the compiler generates for you. It is the dominant data-modelling shape across modern Scala code, and you will see it everywhere in Spark, in standard-library types like `Some` and `Right`, and in your own code.

Adding the `case` keyword to a class gives you, for free:

| Generated | What it does |
|---|---|
| Constructor params are `val` automatically | No need to write `val` on each parameter |
| `apply` on the companion | `Tx(...)` works without `new` |
| `unapply` on the companion | Pattern matching — `case Tx(id, _, amt, _) =>` works |
| Structural `equals` and `hashCode` | Two instances with the same fields are `==` |
| `toString` showing field names | Useful for debugging, logging, REPL |
| `copy` method | Create a new instance with some fields changed |
| `productArity`, `productElement` | Generic iteration over fields |
| Serializable by default | Important for Spark — case classes ship cleanly to executors |

You do not write `new` to construct a case class. You do not write `val` on its parameters. You do not write `equals`, `hashCode`, or `toString`.

In [ ]:
case class Tx(id: String, category: String, amount: Double, status: String)

// Construction — no `new`
val t1 = Tx("T1", "grocery", 42.5, "approved")
val t2 = Tx("T1", "grocery", 42.5, "approved")
val t3 = Tx("T2", "fuel",    58.0, "declined")

// Structural equality — same fields means equal
println(t1 == t2)    // true
println(t1 == t3)    // false

// toString — readable
println(t1)          // Tx(T1,grocery,42.5,approved)

// Fields are val automatically
println(t1.amount)   // 42.5

// copy — update some fields, keep the rest
val t1Approved = t1.copy(status = "declined")
println(t1Approved)  // Tx(T1,grocery,42.5,declined)
println(t1)          // still Tx(T1,grocery,42.5,approved) — original unchanged

## `copy` and named updates — the immutable update idiom

Mutable records use assignment: `tx.status = "declined"`. Immutable records use `.copy`: `tx.copy(status = "declined")`. The original is untouched; you get a new instance with the field changed.

`copy` accepts any subset of the case class's parameters by name. Unsupplied fields default to the original's values. This is the Scala idiom for "update one field of a deeply-nested structure" — combined with `for` comprehensions or lenses libraries, it scales to arbitrary depths.

In [ ]:
case class Address(street: String, city: String, country: String)
case class Person(name: String, age: Int, address: Address)

val ganesh = Person(
  name = "ganesh",
  age = 30,
  address = Address("MG Road", "bengaluru", "india"),
)

// Update one top-level field
val older = ganesh.copy(age = 31)
println(older)

// Update a nested field — copy the inner, then copy the outer with the new inner
val moved = ganesh.copy(address = ganesh.address.copy(city = "chennai"))
println(moved)

// Original is untouched
println(ganesh)

## `case object` — the singleton case

When a case-class-style value has no parameters, you reach for `case object` instead of `case class`. There is only ever one of it; multiple constructions are not meaningful.

This is the canonical shape for sentinel values, status singletons, and the no-payload alternatives in an algebraic data type. Standard-library examples include `Nil` (the empty list) and `None` (the empty option).

In [ ]:
case object Empty

println(Empty)          // Empty
println(Empty == Empty) // true — singletons are equal to themselves by identity

// Singletons compared to other things
case class Filled(value: Int)
val x: Any = Empty
println(x == Empty)     // true
println(x == Filled(1)) // false

## `enum` — Scala 3's first-class ADT

An `enum` declares a closed set of alternatives. Each alternative is a `case`. The compiler knows the set is closed, which means pattern matching on an enum can be checked for exhaustiveness — the compiler warns you when you forget a case.

The simplest form — alternatives with no payload — is the equivalent of a java enum:

```scala
enum Status:
  case Approved, Declined, Pending
```

More interestingly, alternatives can carry data. This is where `enum` becomes the full algebraic data type:

```scala
enum Shape:
  case Circle(radius: Double)
  case Rectangle(width: Double, height: Double)
  case Triangle(base: Double, height: Double)
```

Each case with parameters is essentially a case class. Each case without parameters is essentially a case object.

In [ ]:
// Simple enum — like a java enum
enum Status:
  case Approved, Declined, Pending

val s: Status = Status.Approved
println(s)                       // Approved
println(Status.values.toList)    // List(Approved, Declined, Pending)
println(Status.valueOf("Pending"))  // Pending

// ADT — each case can carry different data
enum Shape:
  case Circle(radius: Double)
  case Rectangle(width: Double, height: Double)
  case Triangle(base: Double, height: Double)

import Shape.*

def area(shape: Shape): Double = shape match
  case Circle(r)       => math.Pi * r * r
  case Rectangle(w, h) => w * h
  case Triangle(b, h)  => 0.5 * b * h

println(area(Circle(3.0)))            // 28.27...
println(area(Rectangle(4.0, 5.0)))    // 20.0
println(area(Triangle(6.0, 4.0)))     // 12.0

## `enum` with parameters and methods

An enum can declare shared parameters and methods that apply to every case. The cases pass values up to those parameters when they construct.

In [ ]:
enum Planet(val massKg: Double, val radiusM: Double):
  case Mercury extends Planet(3.303e23, 2.4397e6)
  case Venus   extends Planet(4.869e24, 6.0518e6)
  case Earth   extends Planet(5.976e24, 6.37814e6)
  case Mars    extends Planet(6.421e23, 3.3972e6)

  // Methods shared across all cases
  private val G = 6.67300e-11
  def surfaceGravity: Double = G * massKg / (radiusM * radiusM)
  def surfaceWeight(otherMass: Double): Double = otherMass * surfaceGravity

println(f"earth gravity = ${Planet.Earth.surfaceGravity}%.2f m/s^2")
println(f"my mass on mars = ${Planet.Mars.surfaceWeight(75)}%.2f N")

## The Scala 2 shape — `sealed trait` + case classes

Before Scala 3 added `enum`, the standard way to express an algebraic data type was a **sealed trait** with case classes (or case objects) extending it. You will still see this everywhere in the wild, including in every Spark source file — Spark targets Scala 2.13.

```scala
sealed trait Status
case object Approved extends Status
case object Declined extends Status
case object Pending  extends Status
```

`sealed` is the magic word — it means *no class outside this file may extend this trait*. The compiler can then enumerate all the subclasses, which is exactly what enables exhaustiveness checking on a match.

The Scala 3 `enum` shape compiles to roughly this under the hood. Use `enum` in new code; recognize the sealed-trait pattern when reading older code or Spark internals.

In [ ]:
// Scala 2 / Spark-style ADT
sealed trait JoinType
case object Inner      extends JoinType
case object LeftOuter  extends JoinType
case object RightOuter extends JoinType
case object FullOuter  extends JoinType
case class  Custom(predicate: String) extends JoinType

def joinSql(j: JoinType): String = j match
  case Inner          => "INNER JOIN"
  case LeftOuter      => "LEFT OUTER JOIN"
  case RightOuter     => "RIGHT OUTER JOIN"
  case FullOuter      => "FULL OUTER JOIN"
  case Custom(pred)   => s"JOIN ON ($pred)"

println(joinSql(Inner))
println(joinSql(Custom("a.id = b.id AND a.ts < b.ts")))

## Pattern matching preview — and why these shapes pair so well

Notebook 06 covers pattern matching in depth. The reason it lives next to this notebook: case classes and enums are precisely the shapes pattern matching is designed to destructure.

A case class declares both the *shape* and the *deconstruction* in one line. An enum declares the *closed set of shapes* in one block. The match expression in your code then matches against those shapes by name, the compiler checks exhaustiveness, and the destructuring pulls fields out by name.

In [ ]:
enum Notification:
  case Email(to: String, subject: String)
  case Sms(to: String, body: String)
  case Push(deviceId: String, title: String, body: String)

import Notification.*

def render(n: Notification): String = n match
  case Email(to, subject)        => s"[EMAIL] $to | $subject"
  case Sms(to, body)             => s"[SMS]   $to | $body"
  case Push(device, title, body) => s"[PUSH]  $device | $title — $body"

val examples = List(
  Email("ganesh@example.com", "welcome"),
  Sms("+91-555-0001", "your OTP is 1234"),
  Push("dev-abc", "alert", "transaction declined"),
)

examples.foreach(n => println(render(n)))